hi


In [1]:
import numpy as np
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery, LeftRightImagery
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from mne.decoding import CSP
from moabb.evaluations import CrossSubjectEvaluation
from sklearn.pipeline import make_pipeline
from scipy import signal
from scipy.io import loadmat
import os
import mne
from scipy.linalg import logm, expm
from sklearn.svm import SVC
from sklearn.metrics import balanced_accuracy_score
from pyriemann.estimation import Covariances
from pyriemann.utils.mean import mean_riemann
from pyriemann.utils.distance import distance_riemann
from sklearn.model_selection import KFold
from tqdm import tqdm

In [2]:
def mdm_classify(cov_train, y_train, cov_test, y_test):
    """
    Minimum Distance to Mean (MDM) classification.

    1) Compute Riemannian mean for each class
    2) Assign each test trial to class with min Riemannian distance
    """
    classes = np.unique(y_train)
    means = {}
    for c in classes:
        means[c] = mean_riemann(cov_train[y_train == c])
    
    predictions = []
    for test_cov in cov_test:
        # Compute distance to each class mean
        dists = {}
        for c in classes:
            dists[c] = distance_riemann(test_cov, means[c])
        # Pick class with minimal distance
        predictions.append(min(dists, key=dists.get))
    
    return np.mean(np.array(predictions) == y_test)

In [3]:
def cov(X):
    covs = np.array([trial @ trial.T for trial in X])
    return covs

In [4]:

def encode_labels(labels_list):
    encoded_list = []
    for labels in labels_list:
        # Create mapping from original labels to 0,1
        unique_labels = np.unique(labels)
        label_map = {unique_labels[i]: i for i in range(len(unique_labels))}
        
        # Apply mapping
        encoded = np.array([label_map[label] for label in labels])
        encoded_list.append(encoded)
    return encoded_list

In [5]:

# Define a causal bandpass filter function using a Butterworth design.
def causal_bandpass_filter(data, lowcut=8, highcut=30, fs=250, order=5):
    nyq = 0.5 * fs
    # Normalize the cutoff frequencies (Matlab's fir1 expects normalized cutoff frequencies
    low = lowcut / nyq
    high = highcut / nyq
    # Design the FIR filter. Note: order+1 coefficients are returned to match Matlab's fir1 which returns n+1 taps.
    b = signal.firwin(order + 1, [low, high], window='hamming', pass_zero=False)
    # Apply the filter causally using lfilter (this introduces a constant delay).
    filtered_data = signal.lfilter(b, [1.0], data)
    return filtered_data


In [6]:
pairs = [
    (0.5, 3.5),
    (0.5, 2.5),
    (0.5, 1.5),
    (1.5, 3.5),
    (1.5, 2.5),
    (2.5, 3.5)
]

In [7]:

def twofour_crosssession(n_classes):
    for tmin, tmax in pairs:

        data_dir = '/home/vishwa/eeg_tl/Recreating papers/BCI2b/BCICIV_2b_gdf'

        # Lists to hold data for all subjects
        train_active_X = []         # List to hold numpy arrays with shape (n_trials, n_channels, n_times) per subject
        train_active_y = []         # List to hold event labels per subject
        train_active_metadata = []  # List to hold event metadata per subject

        # Define subject IDs (B01 to B09)
        subjects = [f'B{subj:02d}' for subj in range(1, 10)]

        for subj in subjects:
            # Define training sessions for this subject (e.g., B0101T.gdf, B0102T.gdf, B0103T.gdf)
            session_ids = ['01T'] #, '02T', '03T']
            subj_epochs_list = []

            for sess in session_ids:
                filename = os.path.join(data_dir, f'{subj}{sess}.gdf')
                
                # Check if file exists to avoid errors
                if not os.path.exists(filename):
                    print(f"File {filename} not found, skipping.")
                    continue
                
                # Load the GDF file
                raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False)
                
                # Extract events from annotations, mapping '769' to 1 (left) and '770' to 2 (right)
                event_id_mapping = {'769': 1, '770': 2}
                events, event_dict = mne.events_from_annotations(raw, event_id=event_id_mapping, verbose=False)
                
                # Select only EEG channels (C3, Cz, C4)
                print(raw.ch_names)
                eeg_channels = [ch for ch in raw.ch_names if ch in ['EEG:C3', 'EEG:Cz', 'EEG:C4']]
                if len(eeg_channels) != 3:
                    print(f"Warning: Expected 3 EEG channels for {subj}{sess}, found {len(eeg_channels)}: {eeg_channels}")
                raw_eeg = raw.pick_channels(eeg_channels, verbose=False)
                
                # Create epochs
                epochs = mne.Epochs(
                    raw_eeg,
                    events,
                    event_id={'left': 1, 'right': 2},
                    tmin=tmin,
                    tmax=tmax,
                    baseline=None,  # No baseline correction, matching your 2a code
                    preload=True,
                    verbose=False
                )
                
                subj_epochs_list.append(epochs)
            
            # Skip subject if no sessions were processed
            if not subj_epochs_list:
                print(f"No valid sessions found for subject {subj}, skipping.")
                continue
            
            # Concatenate epochs across sessions for this subject
            if len(subj_epochs_list) > 1:
                subj_epochs = mne.concatenate_epochs(subj_epochs_list, verbose=False)
            else:
                subj_epochs = subj_epochs_list[0]
            
            # Get the epoch data
            subj_data = subj_epochs.get_data()
            
            # Apply causal bandpass filter to each trial and channel (matching Dataset 2a)
            n_trials, n_channels, n_times = subj_data.shape
            fs = raw.info['sfreq']  # Sampling frequency (250 Hz for Dataset 2b)
            subj_filtered_data = np.empty_like(subj_data)
            for trial in range(n_trials):
                for ch in range(n_channels):
                    subj_filtered_data[trial, ch, :] = causal_bandpass_filter(
                        subj_data[trial, ch, :],
                        lowcut=8,   # Lower bound of sensorimotor rhythm
                        highcut=30, # Upper bound of sensorimotor rhythm
                        fs=fs,
                        order=50    # Filter order
                    )
            
            # Append processed data, labels, and metadata
            train_active_X.append(subj_filtered_data)
            train_active_y.append(subj_epochs.events[:, 2])  # Labels in third column (1 or 2)
            train_active_metadata.append(subj_epochs.events)
            
            # Print shape to verify
            print(f"Subject {subj}: Epoch data shape {subj_filtered_data.shape}")

        print(f"Loaded data for {len(train_active_X)} subjects.")


        # Lists to hold data for all subjects
        eval_active_X = []         # List to hold numpy arrays with shape (n_trials, n_channels, n_times) per subject
        eval_active_y = []         # List to hold event labels per subject
        eval_active_metadata = []  # List to hold event metadata per subject

        # Define subject IDs (B01 to B09)
        subjects = [f'B{subj:02d}' for subj in range(1, 10)]

        for subj in subjects:
            # Define training sessions for this subject (e.g., B0101T.gdf, B0102T.gdf, B0103T.gdf)
            session_ids = ['02T'] #, '02T', '03T']
            subj_epochs_list = []

            for sess in session_ids:
                filename = os.path.join(data_dir, f'{subj}{sess}.gdf')
                
                # Check if file exists to avoid errors
                if not os.path.exists(filename):
                    print(f"File {filename} not found, skipping.")
                    continue
                
                # Load the GDF file
                raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False)
                
                # Extract events from annotations, mapping '769' to 1 (left) and '770' to 2 (right)
                event_id_mapping = {'769': 1, '770': 2}
                events, event_dict = mne.events_from_annotations(raw, event_id=event_id_mapping, verbose=False)
                
                # Select only EEG channels (C3, Cz, C4)
                print(raw.ch_names)
                eeg_channels = [ch for ch in raw.ch_names if ch in ['EEG:C3', 'EEG:Cz', 'EEG:C4']]
                if len(eeg_channels) != 3:
                    print(f"Warning: Expected 3 EEG channels for {subj}{sess}, found {len(eeg_channels)}: {eeg_channels}")
                raw_eeg = raw.pick_channels(eeg_channels, verbose=False)
                
                
                # Create epochs
                epochs = mne.Epochs(
                    raw_eeg,
                    events,
                    event_id={'left': 1, 'right': 2},
                    tmin=tmin,
                    tmax=tmax,
                    baseline=None,  # No baseline correction, matching your 2a code
                    preload=True,
                    verbose=False
                )
                
                subj_epochs_list.append(epochs)
            
            # Skip subject if no sessions were processed
            if not subj_epochs_list:
                print(f"No valid sessions found for subject {subj}, skipping.")
                continue
            
            # Concatenate epochs across sessions for this subject
            if len(subj_epochs_list) > 1:
                subj_epochs = mne.concatenate_epochs(subj_epochs_list, verbose=False)
            else:
                subj_epochs = subj_epochs_list[0]
            
            # Get the epoch data
            subj_data = subj_epochs.get_data()
            
            # Apply causal bandpass filter to each trial and channel (matching Dataset 2a)
            n_trials, n_channels, n_times = subj_data.shape
            fs = raw.info['sfreq']  # Sampling frequency (250 Hz for Dataset 2b)
            subj_filtered_data = np.empty_like(subj_data)
            for trial in range(n_trials):
                for ch in range(n_channels):
                    subj_filtered_data[trial, ch, :] = causal_bandpass_filter(
                        subj_data[trial, ch, :],
                        lowcut=8,   # Lower bound of sensorimotor rhythm
                        highcut=30, # Upper bound of sensorimotor rhythm
                        fs=fs,
                        order=50    # Filter order
                    )
            
            # Append processed data, labels, and metadata
            eval_active_X.append(subj_filtered_data)
            eval_active_y.append(subj_epochs.events[:, 2])  # Labels in third column (1 or 2)
            eval_active_metadata.append(subj_epochs.events)
            
            # Print shape to verify
            print(f"Subject {subj}: Epoch data shape {subj_filtered_data.shape}")

        print(f"Loaded data for {len(train_active_X)} subjects.")
        
        train_active_y = encode_labels(train_active_y)
        eval_active_y = encode_labels(eval_active_y)

        train_active_X = [cov(i) for i in train_active_X]
        eval_active_X = [cov(i) for i in eval_active_X]

        # train_resting_covs = [cov(i) for i in train_active_X]
        # eval_resting_covs =  [cov(i) for i in eval_active_X]

        accuracies = []

        for subj_idx in range(len(train_active_X)):
            # print(subj_idx)
            # Split data into train/test using leave-one-subject-out
            X_test = eval_active_X[subj_idx]
            y_test = eval_active_y[subj_idx]
            
            # Concatenate data from other subjects
            X_train = train_active_X[subj_idx]
            y_train = train_active_y[subj_idx]

            accuracy = mdm_classify(X_train, y_train, X_test, y_test)
            accuracies.append(accuracy)
        print('tmin: ', tmin, ' tmax: ',tmax)
        for subj_idx in range(len(train_active_X)):
            print(f"Subject {subj_idx+1} Test Accuracy: {accuracies[subj_idx]:.2f}")

        print(f"\nMean Cross-Validation Accuracy: {np.mean(accuracies):.2f} ± {np.std(accuracies):.2f}")
    

In [8]:
twofour_crosssession(2)

/tmp/ipykernel_9405/30927458.py:28: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False)


['EEG:C3', 'EEG:Cz', 'EEG:C4', 'EOG:ch01', 'EOG:ch02', 'EOG:ch03']
Subject B01: Epoch data shape (120, 3, 751)


/tmp/ipykernel_9405/30927458.py:28: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False)


['EEG:C3', 'EEG:Cz', 'EEG:C4', 'EOG:ch01', 'EOG:ch02', 'EOG:ch03']
Subject B02: Epoch data shape (120, 3, 751)


/tmp/ipykernel_9405/30927458.py:28: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False)


...

['EEG:C3', 'EEG:Cz', 'EEG:C4', 'EOG:ch01', 'EOG:ch02', 'EOG:ch03']
Subject B07: Epoch data shape (120, 3, 251)


/tmp/ipykernel_9405/30927458.py:116: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False)


['EEG:C3', 'EEG:Cz', 'EEG:C4', 'EOG:ch01', 'EOG:ch02', 'EOG:ch03']
Subject B08: Epoch data shape (120, 3, 251)


/tmp/ipykernel_9405/30927458.py:116: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False)


['EEG:C3', 'EEG:Cz', 'EEG:C4', 'EOG:ch01', 'EOG:ch02', 'EOG:ch03']
Subject B09: Epoch data shape (120, 3, 251)
Loaded data for 9 subjects.
tmin:  2.5  tmax:  3.5
Subject 1 Test Accuracy: 0.52
Subject 2 Test Accuracy: 0.52
Subject 3 Test Accuracy: 0.48
Subject 4 Test Accuracy: 0.66
Subject 5 Test Accuracy: 0.64
Subject 6 Test Accuracy: 0.68
Subject 7 Test Accuracy: 0.51
Subject 8 Test Accuracy: 0.56
Subject 9 Test Accuracy: 0.50

Mean Cross-Validation Accuracy: 0.56 ± 0.07
